# 10 — SAE at Scale: gpt2-medium

This notebook walks through the gpt2-medium SAE run (Investigation #5).
It loads training history and feature analysis from the artifact directory,
plots the loss curve, shows the live/dead feature distribution, and presents
the top-10 features by max activation.

**Run ID**: 51 (spec: `polysemanticity-sae-gpt2-medium`)  
**Model**: gpt2-medium, `blocks.12.hook_resid_pre`, n_features=2048, k=32  
**Corpus**: pile-1k (1000 docs, 8125 effective tokens)

Compare to Investigation #2 (gpt2-small, 128 features, openwebtext sample).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

RUN_ID = 51
ARTIFACT_DIR = repo_root / "artifacts" / f"run-{RUN_ID:06d}"

## Loss Curve

In [ ]:
history = json.loads((ARTIFACT_DIR / "training_history.json").read_text())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history["losses_per_epoch"]) + 1), history["losses_per_epoch"],
        marker="o", color="steelblue", linewidth=2)
ax.axhline(history["initial_loss"], color="gray", linestyle="--", alpha=0.5, label=f"initial loss {history['initial_loss']:.1f}")
ax.axhline(history["final_loss"], color="green", linestyle="--", alpha=0.7, label=f"final loss {history['final_loss']:.2f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean MSE (per-batch average)")
ax.set_title("gpt2-medium SAE Training Loss (run 51)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Initial loss: {history['initial_loss']:.2f}")
print(f"Final loss:   {history['final_loss']:.4f}")
print(f"Reduction:    {100 * (1 - history['final_loss'] / history['initial_loss']):.1f}%")

## Live vs Dead Feature Distribution

In [ ]:
analysis = json.loads((ARTIFACT_DIR / "feature_analysis.json").read_text())
features = analysis["features"]

live = [f for f in features if not f["dead"]]
dead = [f for f in features if f["dead"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar: live vs dead
axes[0].bar(["Live", "Dead"], [len(live), len(dead)], color=["steelblue", "salmon"])
axes[0].set_title(f"Feature Health: {len(live)} live / {len(dead)} dead")
axes[0].set_ylabel("Count")
for i, v in enumerate([len(live), len(dead)]):
    axes[0].text(i, v + 10, f"{v}", ha="center", fontweight="bold")

# Histogram: max activations of live features
max_acts = [f["max_activation"] for f in live]
axes[1].hist(max_acts, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_title("Max Activation Distribution (live features)")
axes[1].set_xlabel("Max Activation")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Live:  {len(live)}  ({100*len(live)/len(features):.1f}%)")
print(f"Dead:  {len(dead)}  ({100*len(dead)/len(features):.1f}%)")

## Coherence Score Distribution

In [ ]:
import statistics

coherences = [f["coherence_score"] for f in live]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(coherences, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(statistics.median(coherences), color="orange", linewidth=2,
           label=f"median={statistics.median(coherences):.3f}")
ax.axvline(statistics.mean(coherences), color="green", linewidth=2, linestyle="--",
           label=f"mean={statistics.mean(coherences):.3f}")
ax.set_xlabel("Jaccard Coherence Score")
ax.set_ylabel("Feature Count")
ax.set_title("Coherence Score Distribution (live features)")
ax.legend()
plt.tight_layout()
plt.show()

coh_1 = sum(1 for c in coherences if c >= 1.0)
coh_gt03 = sum(1 for c in coherences if c > 0.3)
print(f"Coherence=1.0 (fully doc-specialised): {coh_1}")
print(f"Coherence>0.3:                          {coh_gt03}")
print(f"Median:                                 {statistics.median(coherences):.4f}")
print(f"Mean:                                   {statistics.mean(coherences):.4f}")

## Top-10 Features by Max Activation

In [ ]:
top10 = sorted(live, key=lambda f: f["max_activation"], reverse=True)[:10]

rows = []
for f in top10:
    tp = f["top_prompts"]
    first = tp[0]["prompt"][:80] if tp else "(none)"
    rows.append({
        "feature_index": f["feature_index"],
        "max_activation": round(f["max_activation"], 1),
        "coherence": round(f["coherence_score"], 3),
        "top_prompt": first,
    })

df = pd.DataFrame(rows)
df

## Scale Comparison: gpt2-small vs gpt2-medium

| Dimension | gpt2-small (inv. #2) | gpt2-medium (this run) | Scale factor |
|-----------|----------------------|------------------------|--------------|
| d_model | 768 | 1024 | 1.33× |
| n_features | 128 | 2048 | 16× |
| k | 8 | 32 | 4× |
| epochs | 8 | 5 | — |
| max_tokens | 2000 | 8000 | 4× |
| training time | ~40 sec | ~6 min 15 sec | ~9× |
| SAE weights (MB) | ~0.75 MB | ~16 MB | 21× |
| Live features | ~90% | 23% | depends on token budget |

The live-feature ratio drops sharply because 2048 features can't all specialise
with only 8125 tokens available. A 10× token budget (80k tokens, ~20 min training)
would be the next natural experiment to push the dead ratio below 50%.

**Extrapolation to Llama-3.1-8B**:  
gpt2-medium at d_model=1024 took ~6 min.  
Llama-3.1-8B has d_model=4096 (4× larger) and 32 layers (vs 24).  
Activation capture cost scales roughly as O(d_model × n_tokens × n_layers_visited).  
Estimated training time: **3–6 hours overnight** on the same CPU-only path;
**45–90 min with MPS** enabled.

## Scale Report (programmatic summary)

In [ ]:
from mech_interp.analysis.sae_scale_report import generate_scale_report
from mech_interp.config import load_config
from mech_interp.storage import SQLiteResultStore

config = load_config()
store = SQLiteResultStore(config.project.database_path, config.project.artifact_dir)

report = generate_scale_report(RUN_ID, store)
print(json.dumps(report, indent=2))